# SE-LLM-350M — Kaggle Evaluation Notebook

**Run AFTER SFT is complete.**

Evaluates the final SFT model on:
- HumanEval (164 Python coding problems)
- MBPP (500 Python programming problems)
- Manual qualitative tests across multiple languages

**Instructions:**
1. Enable GPU: Settings → Accelerator → P100
2. Add your SFT checkpoint dataset (`se-llm-sft-checkpoints`)
3. Add your tokenizer dataset (`se-llm-data`)
4. Click **Run All**

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
!pip install -q datasets evaluate

In [ ]:
# ── Cell 2: Clone code + link files ───────────────────────────
import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

GITHUB_REPO = 'Abhik2005/se-llm-data'  # ← UPDATE THIS

if not os.path.exists('/kaggle/working/se-llm-350m'):
    try:
        token = secrets.get_secret('GITHUB_TOKEN')
        url   = f'https://{token}@github.com/{GITHUB_REPO}.git'
    except Exception:
        url = f'https://github.com/{GITHUB_REPO}.git'
    !git clone {url} /kaggle/working/se-llm-350m
else:
    !git -C /kaggle/working/se-llm-350m pull

%cd /kaggle/working/se-llm-350m

# Link tokenizer
os.makedirs('tokenizer', exist_ok=True)
tok_src = '/kaggle/input/se-llm-data/tokenizer.json'
tok_dst = 'tokenizer/tokenizer.json'
if os.path.exists(tok_src) and not os.path.exists(tok_dst):
    os.symlink(tok_src, tok_dst)
print('Setup complete')

In [ ]:
# ── Cell 3: Load SFT model ────────────────────────────────────
import os, shutil, torch

SFT_CKPT_DIR = '/kaggle/input/se-llm-sft-checkpoints'
os.makedirs('checkpoints_sft', exist_ok=True)

# Copy checkpoint locally
sft_ckpt = None
for fname in ['sft_final.pt', 'latest.pt']:
    src = os.path.join(SFT_CKPT_DIR, fname)
    if os.path.exists(src):
        dst = f'checkpoints_sft/{fname}'
        shutil.copy(src, dst)
        sft_ckpt = dst
        print(f'Loaded: {fname}')
        break

assert sft_ckpt, 'No SFT checkpoint found! Add the se-llm-sft-checkpoints dataset.'

import sys
sys.path.insert(0, '/kaggle/working/se-llm-350m')
from evaluation.generate import load_model_from_checkpoint, load_tokenizer

device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, cfg = load_model_from_checkpoint(sft_ckpt, device)
tokenizer  = load_tokenizer('tokenizer/tokenizer.json')
print(f'\nModel ready on {device}')

In [ ]:
# ── Cell 4: Qualitative tests ─────────────────────────────────
from evaluation.generate import chat_turn

test_cases = [
    ('Python',     'Write a Python function to check if a string is a palindrome.'),
    ('JavaScript', 'Write a JavaScript function to debounce a function call.'),
    ('SQL',        'Write a SQL query to find all users who placed more than 3 orders in the last 30 days.'),
    ('Java',       'Write a Java method to check if a binary tree is balanced.'),
    ('Code review','Find any bugs in this Python code: def divide(a, b): return a/b'),
    ('Explain',    'What is the difference between a process and a thread?'),
]

print('=== QUALITATIVE EVALUATION ===\n')
for lang, prompt in test_cases:
    print(f'[{lang}] {prompt}')
    response = chat_turn(model, tokenizer, prompt, device=device, max_new_tokens=300)
    print(f'Response:\n{response}')
    print('─' * 60)

In [ ]:
# ── Cell 5: HumanEval benchmark ───────────────────────────────
from evaluation.humaneval import run_humaneval

print('=== HUMANEVAL BENCHMARK ===')
print('Running 164 Python coding problems...\n')

results = run_humaneval(
    checkpoint=sft_ckpt,
    tokenizer_path='tokenizer/tokenizer.json',
    temperature=0.2,          # low temperature for evaluation
    output_file='evaluation/humaneval_results.jsonl',
)

print(f"\nHumanEval pass@1: {results.get('pass@1', 0)*100:.1f}%")

In [ ]:
# ── Cell 6: Perplexity on validation set ──────────────────────
import torch, math
from training.dataset import build_dataloader, estimate_loss

VAL_BIN = '/kaggle/input/se-llm-data/val.bin'

if os.path.exists(VAL_BIN):
    os.makedirs('data/processed', exist_ok=True)
    val_link = 'data/processed/val.bin'
    if not os.path.exists(val_link):
        os.symlink(VAL_BIN, val_link)

    val_loader = build_dataloader(val_link, 2048, batch_size=4, shuffle=False)
    losses     = estimate_loss(model, val_loader, val_loader, eval_batches=50, device=device)
    perplexity = math.exp(losses['val'])

    print(f'Validation loss: {losses["val"]:.4f}')
    print(f'Perplexity:      {perplexity:.2f}')
else:
    print('val.bin not found in dataset — skipping perplexity')

In [ ]:
# ── Cell 7: Print final benchmark summary ─────────────────────
import json

print('\n' + '='*55)
print('  SE-LLM-350M — Final Benchmark Results')
print('='*55)
print(f"  HumanEval pass@1:  {results.get('pass@1', 0)*100:.1f}%")
try:
    print(f'  Validation loss:   {losses["val"]:.4f}')
    print(f'  Perplexity:        {perplexity:.2f}')
except Exception:
    pass
print('='*55)
print('\nInclude these numbers in your model card when publishing the API.')